# 02 — Live Bridge & Tracking

This notebook covers **Gap 2**: taking an offline experiment into production without rewriting anything.

Topics covered:
- `save()` / `load()` — the bridge between notebook and production
- `go_live(traffic_split)` — switching to live mode
- `LiveRouter` — how deterministic traffic splitting works
- `@evalbridge.track` — zero-instrumentation prediction logging
- `evalbridge.log_outcome()` — attaching ground truth later
- `register_experiment()` — connecting decorators to experiments
- Combining offline + live data in a single `evaluate()` call
- `on_confidence(action="promote")` — auto-promotion when ready
- `current_split()` — checking observed traffic ratios

In [1]:
import numpy as np
import evalbridge
from evalbridge import Experiment
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Build two models
X, y = make_classification(n_samples=2000, n_features=20, n_informative=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

baseline_model   = LogisticRegression(random_state=42, max_iter=1000).fit(X_train, y_train)
challenger_model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

base_preds = baseline_model.predict_proba(X_test)[:, 1]
chal_preds = challenger_model.predict_proba(X_test)[:, 1]

print(f"Data ready: {len(X_train)} train / {len(X_test)} test samples")

Data ready: 1200 train / 800 test samples


## Phase 1 — Offline evaluation (notebook)

In [2]:
# Step 1: evaluate offline, just like a normal experiment
exp = Experiment("churn_live_demo", min_samples=100, min_confidence=0.90)
exp.log("baseline",   y_true=y_test.tolist(), y_pred=base_preds.tolist())
exp.log("challenger", y_true=y_test.tolist(), y_pred=chal_preds.tolist())

result_offline = exp.evaluate()
result_offline.summary()

print(f"\nOffline: {len(exp._data['baseline']['y_true'])} baseline samples logged")


┌──────────────┬──────────┬──────────┬────────┬──────────┬─────────────┬────────────┬───┐
│ Model        │ Accuracy │ AUC-ROC  │   F1   │ Log-loss │ Brier score │  Samples   │   │
├──────────────┼──────────┼──────────┼────────┼──────────┼─────────────┼────────────┼───┤
│ baseline     │    0.834 │    0.914 │  0.832 │    0.385 │       0.115 │        800 │    │
│ challenger   │    0.910 │    0.963 │  0.910 │    0.325 │       0.092 │        800 │ ◀ │
└──────────────┴──────────┴──────────┴────────┴──────────┴─────────────┴────────────┴───┘
  Confidence  [████████████████████] 100.0%
  Status      READY ✓
  Winner      challenger


Offline: 800 baseline samples logged


In [3]:
# Step 2: save to disk — this is the bridge
SAVE_PATH = "/tmp/churn_live_demo.evalbridge"
exp.save(SAVE_PATH)
print(f"Experiment saved to {SAVE_PATH}")

Experiment saved to /tmp/churn_live_demo.evalbridge


## Phase 2 — Production (live mode)

Everything below simulates what runs in your production service.
The key point: `Experiment.load()` picks up exactly where the notebook left off.

In [4]:
# Step 3: load in "production"
exp_live = Experiment.load(SAVE_PATH)

print(f"Loaded: {exp_live.name}")
print(f"  baseline samples carried over  : {len(exp_live._data['baseline']['y_true'])}")
print(f"  challenger samples carried over: {len(exp_live._data['challenger']['y_true'])}")

Loaded: churn_live_demo
  baseline samples carried over  : 800
  challenger samples carried over: 800


In [5]:
# Step 4: go live — creates a LiveRouter internally
# traffic_split=0.1 means 10% of requests go to the challenger
exp_live.go_live(traffic_split=0.1)

# The router is now accessible
router = exp_live._router
print(f"Router created: modulo={router._modulo}")
print(f"Every {router._modulo}th request → challenger, rest → baseline")

[evalbridge] 'churn_live_demo' is now live — 10% traffic to challenger
Router created: modulo=10
Every 10th request → challenger, rest → baseline


## How LiveRouter works internally

In [6]:
from evalbridge.bridge import LiveRouter

# Show the routing pattern for the first 25 requests
demo_router = LiveRouter(traffic_split=0.1)

results = [demo_router.route() for _ in range(25)]
print("Routing decisions for first 25 requests:")
for i, decision in enumerate(results, 1):
    marker = "← CHALLENGER" if decision == "challenger" else ""
    print(f"  request {i:>2}: {decision} {marker}")

split = demo_router.current_split()
print(f"\nObserved split: baseline={split['baseline']:.1%}  challenger={split['challenger']:.1%}")

Routing decisions for first 25 requests:
  request  1: baseline 
  request  2: baseline 
  request  3: baseline 
  request  4: baseline 
  request  5: baseline 
  request  6: baseline 
  request  7: baseline 
  request  8: baseline 
  request  9: baseline 
  request 10: challenger ← CHALLENGER
  request 11: baseline 
  request 12: baseline 
  request 13: baseline 
  request 14: baseline 
  request 15: baseline 
  request 16: baseline 
  request 17: baseline 
  request 18: baseline 
  request 19: baseline 
  request 20: challenger ← CHALLENGER
  request 21: baseline 
  request 22: baseline 
  request 23: baseline 
  request 24: baseline 
  request 25: baseline 

Observed split: baseline=92.0%  challenger=8.0%


## @evalbridge.track — zero-instrumentation logging

In [7]:
# Step 5: register the experiment so @track can find it by name
evalbridge.register_experiment(exp_live)

# Step 6: decorate prediction functions
# The functions themselves are unchanged — @track wraps them invisibly
@evalbridge.track(experiment="churn_live_demo", model="challenger")
def predict_challenger(features):
    return float(challenger_model.predict_proba([features])[0, 1])

@evalbridge.track(experiment="churn_live_demo", model="baseline")
def predict_baseline(features):
    return float(baseline_model.predict_proba([features])[0, 1])

# Demo: a single tracked prediction
pred = predict_challenger(X_test[0])
rid  = predict_challenger.last_request_id

print(f"Prediction: {pred:.4f}")
print(f"Request ID: {rid}")
print(f"Stored in _pending: {rid[:8]}... → pred={pred:.4f}")

# Attach ground truth immediately (in real life this comes hours later)
evalbridge.log_outcome(request_id=rid, y_true=int(y_test[0]))
print(f"\nGround truth logged. _pending size now: {len(evalbridge.decorators._pending)}")
print(f"challenger samples in exp_live: {len(exp_live._data['challenger']['y_true'])}")

Prediction: 0.2600
Request ID: a3ade1ea-cc5f-4610-8f4e-a1c8f6062860
Stored in _pending: a3ade1ea... → pred=0.2600

Ground truth logged. _pending size now: 0
challenger samples in exp_live: 801


## Simulate 300 live requests

In [8]:
import random
random.seed(0)

n_live = 300
request_ids = []
ground_truths = []

for i in range(n_live):
    idx = i % len(X_test)
    features = X_test[idx]
    true_label = int(y_test[idx])

    # Router decides which model serves this request
    model_name = router.route()

    if model_name == "challenger":
        predict_challenger(features)
        rid = predict_challenger.last_request_id
    else:
        predict_baseline(features)
        rid = predict_baseline.last_request_id

    request_ids.append(rid)
    ground_truths.append(true_label)

# Simulate ground truth arriving in a batch (e.g. next day's labels)
for rid, gt in zip(request_ids, ground_truths):
    evalbridge.log_outcome(request_id=rid, y_true=gt)

split = router.current_split()
print(f"Live requests routed: {n_live}")
print(f"  → baseline   : {int(split['baseline'] * n_live)} requests ({split['baseline']:.1%})")
print(f"  → challenger : {int(split['challenger'] * n_live)} requests ({split['challenger']:.1%})")
print(f"\nTotal samples in experiment:")
print(f"  baseline  : {len(exp_live._data['baseline']['y_true'])} (offline + live)")
print(f"  challenger: {len(exp_live._data['challenger']['y_true'])} (offline + live)")

Live requests routed: 300
  → baseline   : 270 requests (90.0%)
  → challenger : 30 requests (10.0%)

Total samples in experiment:
  baseline  : 1070 (offline + live)
  challenger: 831 (offline + live)


## Evaluate with combined offline + live data

In [9]:
result_live = exp_live.evaluate()
result_live.summary()


┌──────────────┬──────────┬──────────┬────────┬──────────┬─────────────┬────────────┬───┐
│ Model        │ Accuracy │ AUC-ROC  │   F1   │ Log-loss │ Brier score │  Samples   │   │
├──────────────┼──────────┼──────────┼────────┼──────────┼─────────────┼────────────┼───┤
│ baseline     │    0.823 │    0.909 │  0.823 │    0.392 │       0.119 │      1,070 │    │
│ challenger   │    0.907 │    0.961 │  0.906 │    0.326 │       0.093 │        831 │ ◀ │
└──────────────┴──────────┴──────────┴────────┴──────────┴─────────────┴────────────┴───┘
  Confidence  [████████████████████] 100.0%
  Status      READY ✓
  Winner      challenger



## on_confidence — auto-promote when ready

In [10]:
promoted = []

# Register a callback that fires when confidence >= 0.90
exp_live.on_confidence(
    threshold=0.90,
    action=lambda r: promoted.append(r.winner)
)

# evaluate() fires the callback automatically
result_auto = exp_live.evaluate()

if promoted:
    print(f"Auto-promote fired! Winner '{promoted[-1]}' would be promoted to Production.")
else:
    print("Confidence threshold not yet reached.")

print(f"Confidence: {result_auto.confidence:.4f}")

Auto-promote fired! Winner 'challenger' would be promoted to Production.
Confidence: 1.0000


## Different traffic splits — effect on challenger exposure

In [11]:
print(f"{'split':>8} {'modulo':>8} {'challenger/100 requests':>25}")
print("-" * 45)
for split in [0.05, 0.10, 0.20, 0.25, 0.50]:
    r = LiveRouter(split)
    routes = [r.route() for _ in range(100)]
    n_chal = sum(1 for x in routes if x == "challenger")
    print(f"{split:>8.0%} {r._modulo:>8}  {n_chal:>25} requests")

import os
os.remove(SAVE_PATH)

   split   modulo   challenger/100 requests
---------------------------------------------
      5%       20                          5 requests
     10%       10                         10 requests
     20%        5                         20 requests
     25%        4                         25 requests
     50%        2                         50 requests
